# TravelMind on AWS Agent Builder: Console and Code, From Scratch

You already built this agent by hand. You wrote the loop that calls the model, reads `tool_use`, runs the tool, feeds the result back, and repeats. Today you hand that loop to AWS.

**The one idea for the whole notebook:** an agent runtime is just that while-loop. Agent Builder runs it for you. You describe a model, instructions, and tools, and AWS does the orchestration.

We will do it three ways, in order:
1. understand the mapping from your code to the console
2. build it by clicking in the console
3. call it from code, then build the entire thing from code

```mermaid
flowchart LR
    A["Understand the mapping"] --> B["Build by clicking"]
    B --> C["Invoke from code"]
    C --> D["Build fully from code"]
    D --> E["Clean up and compare"]
```

> **One honest warning.** The service behind Agent Builder is now called Amazon Bedrock Agents Classic and closes to new customers on July 30, 2026. Existing accounts keep working. The successor is Amazon Bedrock AgentCore, which runs agents from any framework, including your Strands agents. Learn the shape here; it transfers.

## Setup

VS Code: make a folder, open it, create a venv, activate, run the install cell, then `aws configure` with region `us-east-1`.

This notebook touches several services. To run the full programmatic build later you need permissions for Bedrock Agents, Lambda, and IAM. If you only want to invoke a console-built agent, you need far less.

In [ ]:
%pip install -q boto3

In [ ]:
import json, time, io, zipfile, uuid
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

REGION = "us-east-1"
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # your anchor, as an inference profile
cfg = Config(retries={"max_attempts": 5, "mode": "adaptive"})

# The clients you need. Note there are THREE different Bedrock clients.
bedrock          = boto3.client("bedrock-runtime", region_name=REGION, config=cfg)   # models
bedrock_agent    = boto3.client("bedrock-agent", region_name=REGION, config=cfg)     # build agents
bedrock_agent_rt = boto3.client("bedrock-agent-runtime", region_name=REGION, config=cfg)  # call agents
lambda_client    = boto3.client("lambda", region_name=REGION, config=cfg)
iam              = boto3.client("iam", region_name=REGION, config=cfg)
sts              = boto3.client("sts", region_name=REGION, config=cfg)

ACCOUNT_ID = sts.get_caller_identity()["Account"]
SUFFIX = uuid.uuid4().hex[:6]        # keeps names unique so re-runs do not collide
print("account:", ACCOUNT_ID, "region:", REGION, "run suffix:", SUFFIX)

## The mapping: your code becomes their console

Everything you hand-wrote has a home in Agent Builder. This is the intuition for the whole notebook.

| Your hand-built agent | Agent Builder equivalent |
|---|---|
| `MODEL_ID` | the agent's foundation model |
| your `system` prompt | the agent Instructions |
| a `toolSpec` (name, schema) | an action group function definition |
| your `dispatch` plus tool functions | a Lambda function |
| the `run_agent` while-loop | managed orchestration, invisible to you |

```mermaid
flowchart LR
    U["user message"] --> M["model call"]
    M --> S{"tool_use?"}
    S -->|yes| T["Lambda runs the tool"]
    T --> R["result back to the model"]
    R --> M
    S -->|no| O["final answer"]
```

The loop is identical to yours. The only change is who runs it.

## Part B: the tool code, as a Lambda

The agent reasons but never runs your code. When it decides to call `check_entitlements(delay_hours=7, fare_class="FLEX")`, a Lambda function executes it. The Lambda is your old `dispatch` plus tool functions, living in AWS.

### The Lambda contract

Bedrock sends a fixed event and expects a fixed response. This is where most first agents break.

> **What goes wrong (the number one error).** "The server encountered an error processing the Lambda response." Three causes, all in the response you return:
> - `body` must be a **string**. Use `json.dumps(result)`, never a raw dict.
> - Content type is `TEXT` for function-schema tools. Match it exactly.
> - Echo back the same `actionGroup` and `function` you received.
>
> One more: every incoming parameter arrives as a **string**, even numbers. Cast them yourself.

In [ ]:
# One source of truth for the handler: we exec it locally to test, and write it to disk to deploy.
HANDLER_CODE = r"""
import json

def _params(event):
    # Every parameter arrives as a string. Return a name -> value dict.
    return {p["name"]: p["value"] for p in event.get("parameters", [])}

def check_entitlements(delay_hours, fare_class):
    d = float(delay_hours)                       # cast: it arrived as a string
    return {"meal_voucher": d >= 2, "hotel_voucher": d >= 6, "fare_class": fare_class}

def get_booking(pnr):
    if pnr != "JX48Q2":
        return {"error": "PNR not found. Ask the passenger to recheck the code."}
    return {"pnr": pnr, "origin": "BLR", "destination": "SIN", "fare_class": "FLEX",
            "original_flight": "TM482", "status": "CANCELLED"}

DISPATCH = {"check_entitlements": check_entitlements, "get_booking": get_booking}

def lambda_handler(event, context):
    action_group = event["actionGroup"]
    function = event["function"]
    args = _params(event)
    fn = DISPATCH.get(function)
    result = fn(**args) if fn else {"error": f"unknown function {function}"}
    # THE CONTRACT: string body, TEXT content type, echo actionGroup and function.
    return {
        "messageVersion": "1.0",
        "response": {
            "actionGroup": action_group,
            "function": function,
            "functionResponse": {"responseBody": {"TEXT": {"body": json.dumps(result)}}},
        },
    }
"""
print(HANDLER_CODE)

### Prove the contract locally before touching AWS

Simulate the exact event Bedrock sends, run the handler, and check the response shape. If this passes, the Lambda will not throw the response error.

In [ ]:
exec(HANDLER_CODE, globals())    # makes lambda_handler callable here

fake_event = {
    "messageVersion": "1.0",
    "actionGroup": "travelmind-actions",
    "function": "check_entitlements",
    "parameters": [
        {"name": "delay_hours", "type": "number", "value": "7"},   # note: string value
        {"name": "fare_class", "type": "string", "value": "FLEX"},
    ],
}
out = lambda_handler(fake_event, None)
print(json.dumps(out, indent=2))

body = out["response"]["functionResponse"]["responseBody"]["TEXT"]["body"]
assert isinstance(body, str), "body MUST be a string"
parsed = json.loads(body)
assert parsed["meal_voucher"] is True and parsed["hotel_voucher"] is True   # 7h -> both
print("\ncontract OK, parsed result:", parsed)

## Part C: build it by clicking (the console)

The "Agent Builder" experience is the console. Exact steps, from scratch.

1. Bedrock console, left nav, **Builder tools**, then **Agents**, then **Create agent**.
2. Name it `travelmind-desk`, choose **Create**. The Agent builder pane opens.
3. Agent resource role: **Create and use a new service role**. This wires Door 1 for you.
4. Select model: your Haiku 4.5 inference profile.
5. Instructions: paste the TravelMind policy from your production notebook.
6. Choose **Save**, then the **Action groups** tab, then **Add**.
7. Name it `travelmind-actions`, type **Define with function details**.
8. Action group invocation: **Quick create a new Lambda function**. This wires Door 2 for you.
9. Add a function `check_entitlements` with parameters `delay_hours` (number, required) and `fare_class` (string, required).
10. Save. Open the created Lambda, paste the handler above, deploy it.
11. Back on the agent, **Save**, then **Prepare**. Prepare compiles the draft. Do it after every change.
12. Test in the built-in window. Ask about a seven hour delay on a FLEX fare.

```mermaid
flowchart TD
    A["Create agent"] --> B["Pick model plus instructions"]
    B --> C["Add action group, function details"]
    C --> D["Quick create Lambda, paste handler"]
    D --> E["Save then Prepare"]
    E --> F["Test in the console"]
```

> **What goes wrong.** If the test window says the Lambda response failed, you broke the contract from Part B. If it says access denied to the model, you have not enabled model access for that model in this region.

## Part D: talk to it from code

Once your console agent has an alias, your application calls it in a few lines. The reply comes back as a **stream** of chunks that you join.

In [ ]:
def invoke_agent(agent_id, alias_id, prompt, session_id=None, trace=False):
    session_id = session_id or uuid.uuid4().hex
    resp = bedrock_agent_rt.invoke_agent(
        agentId=agent_id, agentAliasId=alias_id,
        sessionId=session_id, inputText=prompt, enableTrace=trace)
    answer = ""
    for event in resp["completion"]:
        if "chunk" in event:
            answer += event["chunk"]["bytes"].decode()
        elif trace and "trace" in event:
            # The agent's step by step reasoning. Your debugger for agents.
            t = event["trace"]["trace"]
            print("  [trace]", list(t.keys()))
    return answer, session_id

# Fill these from your console agent (Agent overview shows the ID; create an alias to get its ID).
CONSOLE_AGENT_ID = "REPLACE_ME"
CONSOLE_ALIAS_ID = "REPLACE_ME"   # or "TSTALIASID" to hit the draft you tested in the console

if CONSOLE_AGENT_ID != "REPLACE_ME":
    text, sid = invoke_agent(CONSOLE_AGENT_ID, CONSOLE_ALIAS_ID,
                             "Delay is 7 hours on a FLEX fare. What am I owed?", trace=True)
    print(text)
else:
    print("Set CONSOLE_AGENT_ID and CONSOLE_ALIAS_ID from your console agent to run this.")

> **What goes wrong.** `dependencyFailedException: Access denied when calling Bedrock` usually means the agent service role is missing `bedrock:InvokeModel` for the model, or model access is not enabled, or the agent was never prepared. `sessionId` is the memory handle: reuse it to continue a conversation, change it to start fresh. `TSTALIASID` hits the draft; a real alias hits a frozen version.

## Part E: build the entire thing from code

Clicking is for learning. Real teams build agents as code so they are repeatable. We now create everything: two IAM roles, a Lambda, the agent, the action group, then prepare and alias and invoke.

```mermaid
flowchart TD
    R1["1. Lambda execution role"] --> L["2. Deploy Lambda"]
    L --> R2["3. Agent service role, Door 1"]
    R2 --> AG["4. Create agent"]
    AG --> P2["5. Lambda resource policy, Door 2, needs agent ARN"]
    P2 --> ACT["6. Action group with functionSchema"]
    ACT --> PREP["7. Prepare then alias"]
    PREP --> INV["8. Invoke"]
```

The two permission doors, made concrete:

```mermaid
flowchart LR
    subgraph D1["Door 1: agent service role"]
        AR["role"] --> IM["bedrock InvokeModel"]
        AR --> LI["lambda InvokeFunction"]
    end
    subgraph D2["Door 2: Lambda resource policy"]
        LP["policy"] --> OK["allow bedrock.amazonaws.com, scoped to this agent"]
    end
```

> **Warning.** The cells below create real AWS resources and can cost money. Run them deliberately. Part F cleans everything up.

In [ ]:
# Shared names and a generic waiter used across the build.
AGENT_NAME = f"travelmind-desk-{SUFFIX}"
FUNC_NAME  = f"travelmind-tools-{SUFFIX}"
LAMBDA_ROLE = f"travelmind-lambda-role-{SUFFIX}"
AGENT_ROLE  = f"travelmind-agent-role-{SUFFIX}"
ACTION_GROUP = "travelmind-actions"

def wait_until(get_status, target, timeout=120, interval=4, label="resource"):
    end = time.time() + timeout
    s = None
    while time.time() < end:
        s = get_status()
        print(f"  {label}: {s}")
        if s in target:
            return s
        time.sleep(interval)
    raise TimeoutError(f"{label} stuck at {s}, wanted {target}")

### E1. The Lambda execution role

Every Lambda needs a role that lets it write logs. This is separate from the agent's role.

In [ ]:
lambda_trust = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"},
    "Action": "sts:AssumeRole"}]}

r = iam.create_role(RoleName=LAMBDA_ROLE,
                    AssumeRolePolicyDocument=json.dumps(lambda_trust),
                    Description="TravelMind Lambda execution role")
LAMBDA_ROLE_ARN = r["Role"]["Arn"]
iam.attach_role_policy(RoleName=LAMBDA_ROLE,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
print("lambda role:", LAMBDA_ROLE_ARN)
time.sleep(10)   # IAM is eventually consistent; give the role a moment to propagate

### E2. Deploy the Lambda

We zip the handler from Part B in memory and create the function.

In [ ]:
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("lambda_function.py", HANDLER_CODE)
buf.seek(0)

# Retry once, since a fresh IAM role can lag behind create_function.
for attempt in range(3):
    try:
        fn = lambda_client.create_function(
            FunctionName=FUNC_NAME, Runtime="python3.12", Role=LAMBDA_ROLE_ARN,
            Handler="lambda_function.lambda_handler",
            Code={"ZipFile": buf.getvalue()}, Timeout=30, MemorySize=128,
            Description="TravelMind tools for the Bedrock agent")
        break
    except ClientError as e:
        if "cannot be assumed" in str(e) and attempt < 2:
            print("  role not ready, retrying..."); time.sleep(8); continue
        raise
LAMBDA_ARN = fn["FunctionArn"]
wait_until(lambda _=None: lambda_client.get_function(FunctionName=FUNC_NAME)["Configuration"]["State"],
           {"Active"}, label="lambda")
print("lambda:", LAMBDA_ARN)

### E3. The agent service role, Door 1

This role lets the agent invoke the model and invoke the Lambda.

> **Production.** Scope the model resource to the specific model ARNs, not a wildcard. For a cross-region inference profile you allow the profile ARN plus the underlying foundation-model ARNs across regions.

In [ ]:
IP_ARN = f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:inference-profile/{MODEL_ID}"
FM_ARN = "arn:aws:bedrock:*::foundation-model/anthropic.claude-haiku-4-5-20251001-v1:0"

agent_trust = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"},
    "Action": "sts:AssumeRole",
    "Condition": {"StringEquals": {"aws:SourceAccount": ACCOUNT_ID}}}]}

agent_perm = {"Version": "2012-10-17", "Statement": [
    {"Sid": "InvokeModel", "Effect": "Allow",
     "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
     "Resource": [IP_ARN, FM_ARN]},
    {"Sid": "InvokeLambda", "Effect": "Allow",
     "Action": "lambda:InvokeFunction", "Resource": LAMBDA_ARN}]}

r = iam.create_role(RoleName=AGENT_ROLE,
                    AssumeRolePolicyDocument=json.dumps(agent_trust),
                    Description="TravelMind Bedrock agent service role")
AGENT_ROLE_ARN = r["Role"]["Arn"]
iam.put_role_policy(RoleName=AGENT_ROLE, PolicyName="agent-invoke",
                    PolicyDocument=json.dumps(agent_perm))
print("agent role:", AGENT_ROLE_ARN)
time.sleep(10)

### E4. Create the agent

> **What goes wrong.** Newer models are only reachable through a cross-region inference profile, so pass the `us.` profile ID (or its ARN). A bare model ID can be rejected. After creation the agent is `NOT_PREPARED`; that is expected.

In [ ]:
INSTRUCTIONS = (
    "You are TravelMind's disruption assistant. When a passenger reports a delay, use the "
    "check_entitlements tool to decide meal and hotel vouchers from the delay hours and fare "
    "class. Never guess entitlements. Report exactly what the tool returns, warmly and briefly. "
    "You cannot rebook or charge anything; present options for the passenger to confirm.")

a = bedrock_agent.create_agent(
    agentName=AGENT_NAME, agentResourceRoleArn=AGENT_ROLE_ARN,
    foundationModel=MODEL_ID, instruction=INSTRUCTIONS, idleSessionTTLInSeconds=600)
AGENT_ID = a["agent"]["agentId"]
AGENT_ARN = a["agent"]["agentArn"]
wait_until(lambda _=None: bedrock_agent.get_agent(agentId=AGENT_ID)["agent"]["agentStatus"],
           {"NOT_PREPARED"}, label="agent")
print("agent id:", AGENT_ID)

### E5. Door 2, the Lambda resource policy

Now that the agent exists, allow it to invoke the Lambda. We scope it to this exact agent ARN.

> **Production.** The `SourceArn` condition is what stops every other Bedrock agent in the account from invoking your Lambda. Never skip it.

In [ ]:
lambda_client.add_permission(
    FunctionName=FUNC_NAME, StatementId="allow-bedrock-agent",
    Action="lambda:InvokeFunction", Principal="bedrock.amazonaws.com",
    SourceArn=AGENT_ARN, SourceAccount=ACCOUNT_ID)
print("Door 2 open: this agent may now invoke the Lambda.")

### E6. The action group

This is your `toolSpec`, expressed as a `functionSchema`, wired to the Lambda. Add more functions to the `functions` list the same way.

In [ ]:
bedrock_agent.create_agent_action_group(
    agentId=AGENT_ID, agentVersion="DRAFT", actionGroupName=ACTION_GROUP,
    actionGroupState="ENABLED",
    actionGroupExecutor={"lambda": LAMBDA_ARN},
    functionSchema={"functions": [{
        "name": "check_entitlements",
        "description": "Decide meal and hotel voucher eligibility from delay hours and fare class.",
        "parameters": {
            "delay_hours": {"type": "number", "description": "Hours of delay", "required": True},
            "fare_class": {"type": "string", "description": "Fare class such as FLEX", "required": True}},
    }]})
print("action group created and wired to the Lambda.")

### E7. Prepare, then alias

> **What goes wrong.** Forget `prepare_agent` and your action group simply does not exist to the running agent. Prepare compiles the draft. Then an alias gives you a stable target to invoke.

In [ ]:
bedrock_agent.prepare_agent(agentId=AGENT_ID)
wait_until(lambda _=None: bedrock_agent.get_agent(agentId=AGENT_ID)["agent"]["agentStatus"],
           {"PREPARED"}, label="agent prepare")

alias = bedrock_agent.create_agent_alias(agentAliasName="live", agentId=AGENT_ID)
ALIAS_ID = alias["agentAlias"]["agentAliasId"]
wait_until(lambda _=None: bedrock_agent.get_agent_alias(
               agentId=AGENT_ID, agentAliasId=ALIAS_ID)["agentAlias"]["agentAliasStatus"],
           {"PREPARED"}, label="alias")
print("alias id:", ALIAS_ID)

### E8. Invoke the agent you built entirely in code

In [ ]:
text, sid = invoke_agent(AGENT_ID, ALIAS_ID,
    "My flight was cancelled and I've been delayed 7 hours on a FLEX fare. What am I owed?",
    trace=True)
print("\nAGENT:", text)

You just reproduced your hand-built agent as fully managed infrastructure. The Lambda ran your tool, the managed loop called it, and you never wrote `run_agent`.

## Part F: clean up

Delete in reverse order of creation so nothing is left billing you. IAM roles need their policies removed first.

In [ ]:
def safe(fn, *a, **k):
    try:
        fn(*a, **k); return True
    except ClientError as e:
        print("  skip:", e.response["Error"]["Code"]); return False

# alias and agent
safe(bedrock_agent.delete_agent_alias, agentId=AGENT_ID, agentAliasId=ALIAS_ID)
safe(bedrock_agent.delete_agent, agentId=AGENT_ID, skipResourceInUseCheck=True)

# lambda
safe(lambda_client.delete_function, FunctionName=FUNC_NAME)

# agent role: remove inline policy, then delete
safe(iam.delete_role_policy, RoleName=AGENT_ROLE, PolicyName="agent-invoke")
safe(iam.delete_role, RoleName=AGENT_ROLE)

# lambda role: detach managed policy, then delete
safe(iam.detach_role_policy, RoleName=LAMBDA_ROLE,
     PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
safe(iam.delete_role, RoleName=LAMBDA_ROLE)
print("cleanup done.")

## Part G: connect the dots

You have now run the same agent loop three ways.

```mermaid
flowchart LR
    H["Hand-built loop: most control, most code, best for understanding"] --> M["Agent Builder: AWS runs the loop, almost no code, Classic is closing"]
    M --> AC["AgentCore: any framework, production scale, the forward path"]
```

| | Who runs the loop | Your code | Best for |
|---|---|---|---|
| Hand-built | you | the whole loop | learning, full control, custom orchestration |
| Agent Builder | AWS managed | tools plus config | quick internal tools, learning managed agents |
| AgentCore | AWS managed | your framework agent | taking Strands agents to production |

The skills transfer across all three, and they are the ones you already have:

- tools are contracts: a name, typed parameters, and a returned result
- the model chooses the tool and arguments, code executes them
- permissions are a two-way door, and people forget the second one
- every irreversible action stays gated, by omitting the tool

> **Production checklist.** Scope every IAM policy to specific ARNs. Add the `SourceArn` condition on the Lambda. Grant callers only `bedrock:InvokeAgent` on one alias. Respect the 15 minute Lambda ceiling and use Return of Control for longer tasks. Turn on traces, watch token cost, set a budget alarm, and plan your move to AgentCore.

Learn the loop once. Run it anywhere.